# Arbimon Audio Analyzer

This notebook downloads recordings from your **Arbimon** projects and runs an external **Neural Network Classification Model** to detect specific species in each recording.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read/save files
2. **Installs the necessary software** automatically
3. **Logs in to Arbimon** so it can access your recordings
4. **Downloads audio** from the recording sites and time windows you choose
5. **Loads a model** from HuggingFace or your Google Drive (e.g. BirdNET, Perch, custom)
6. **Analyzes every recording** and saves the results as text files on your Drive

### Before you start:
- You need a **Google account** with Google Drive
- You need an **Arbimon account** (at [arbimon.org](https://arbimon.org)) with access to your project
- You need the **stream IDs** for each recording site (see Step 3 for how to find them)
- You need a **TFLite (`.tflite`) or ONNX (`.onnx`) model file** and a matching **labels file** (`.txt`, one species per line)

### How to run:
Run each cell **one at a time**, from top to bottom. Click the ▶ button on the left of each cell, or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running, it shows `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you exactly what to fix.

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit [biodiversica.xyz](https://biodiversica.xyz).

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl \
    -O /tmp/rfcx-0.3.1-py3-none-any.whl

!pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
!pip install ai-edge-litert onnxruntime librosa soundfile huggingface_hub -q

print('\nAll packages installed successfully.')

---
## Step 2 — Log in to Arbimon

The next cell will log you in to the Arbimon / RFCx platform.

**What to expect:**
- The first time you run this, a **URL and a short code** will appear. Open that URL in your browser and enter the code to authorize access. This is the same account you use to log in at [arbimon.org](https://arbimon.org).
- Your login credentials will be **saved to your Google Drive** at the path you set below, so future runs skip this step automatically.

**Set the path where you want to save your credentials on Google Drive:**

In [ ]:
#@title 🔑 Log in to Arbimon { display-mode: "form" }
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os
import rfcx

os.makedirs(os.path.dirname(CREDENTIALS_PATH), exist_ok=True)

client = rfcx.Client()

if os.path.exists(CREDENTIALS_PATH):
    print('Credentials file found — logging in automatically...')
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
else:
    print('No saved credentials found.')
    print('A URL will appear below. Open it in your browser and log in with your Arbimon account.')
    print('-' * 70)
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('-' * 70)
    print(f'Credentials saved to: {CREDENTIALS_PATH}')
    print('Next time, login will happen automatically.')

print('\nLogged in to Arbimon successfully.')

### Optional: Find your Arbimon stream IDs

If you do not know the **stream ID** for each recording site in your Arbimon project, run the cell below. It will list all projects and sites available to your account.

> You will need the **stream ID** (the short code next to each site name) to fill in the configuration in Step 3.

In [ ]:
#@title 🔍 List my Arbimon projects and sites { display-mode: "form" }

print('Fetching your Arbimon projects and recording sites...')
print('=' * 70)

projects = client.projects()
if not projects:
    print('No projects found. Make sure you are logged in to the correct account.')
else:
    for project in projects:
        print(f"\nPROJECT: {project['name']}  (id: {project['id']})")
        print('  Recording sites (streams):')
        try:
            streams = client.streams(projects=project['id'])
            if streams:
                for stream in streams:
                    print(f"    stream_id: '{stream['id']}',  name: '{stream['name']}'")
            else:
                print('    (no sites found in this project)')
        except Exception as e:
            print(f'    (could not load sites: {e})')

---
## Step 3 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in each form and run the cell.

Each form cell has a small ▶ button on the left. Run them all from top to bottom:
1. **General Settings** — confidence threshold, results folder
2. **Model** — where your model is stored and how it works
3. **Recording Sites** — one form per site (up to 4); leave empty to skip

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [ ]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Results folder** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/arbimon_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Detection threshold** — minimum confidence score to save a detection (0.0–1.0).
#@markdown Lower = more detections (may include false positives). Higher = fewer but more reliable.
MIN_CONFIDENCE = 0.25  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Top K detections per segment** — maximum number of detections to keep per audio segment.
#@markdown 1 = only the highest-confidence detection per segment. Increase to keep more.
TOP_K = 1  #@param {type:"integer"}

AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results folder  : {DRIVE_RESULTS_DIR}")
print(f"Min. confidence : {MIN_CONFIDENCE}")
print(f"Top K           : {TOP_K}")

In [ ]:
#@title ⚙️ Audio Preprocessing { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown
#@markdown ---
#@markdown **Frequency filter** — remove frequencies outside the range of interest.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Low-cut frequency (Hz)** — used for highpass and bandpass filters.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **High-cut frequency (Hz)** — used for lowpass and bandpass filters.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Playback speed** — 1.0 = normal. Below 1.0 slows down and lengthens audio;
#@markdown above 1.0 speeds up and shortens it. Useful for shifting frequency content
#@markdown into the model's expected range (e.g. 0.5x halves all frequencies).
AUDIO_SPEED = 1.0  #@param {type:"slider", min:0.25, max:4.0, step:0.05}

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Preprocessing enabled:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Preprocessing : none')

#@markdown ---
#@markdown **Time of day filter** — only analyze recordings within a specific daily time window.
#@markdown Leave disabled to analyze all recordings regardless of time.
TIME_FILTER_ENABLED = False  #@param {type:"boolean"}
#@markdown **Window start (HH:MM)**
TIME_FILTER_START = "06:00"  #@param {type:"string"}
#@markdown **Window end (HH:MM)** — can cross midnight, e.g. 22:00 to 06:00.
TIME_FILTER_END = "10:00"  #@param {type:"string"}

if TIME_FILTER_ENABLED:
    print(f'Time filter : {TIME_FILTER_START} – {TIME_FILTER_END}')

In [ ]:
#@title 🤖 Model { display-mode: "form" }

#@markdown **Model source** — where is your model file stored?
MODEL_SOURCE = "huggingface"  #@param ["huggingface", "google_drive"]

#@markdown ---
#@markdown ### If model source = Google Drive
#@markdown Full path to the model file on your Drive (`.tflite` or `.onnx`).
#@markdown Example: `/content/drive/MyDrive/models/frog_detector.tflite`
DRIVE_MODEL_PATH  = "/content/drive/MyDrive/models/my_model.tflite"  #@param {type:"string"}
#@markdown Full path to the labels file on your Drive.
DRIVE_LABELS_PATH = "/content/drive/MyDrive/models/labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### If model source = HuggingFace
#@markdown The repo ID is the part after `huggingface.co/` in the model URL.
#@markdown Default: `justinchuby/BirdNET-onnx` (BirdNET v2.4 in ONNX format)
HF_REPO_ID     = "justinchuby/BirdNET-onnx"  #@param {type:"string"}
HF_MODEL_FILE  = "model.onnx"              #@param {type:"string"}
#@markdown The labels file can be in a **different** HuggingFace repo. Leave blank to use the same repo as the model.
HF_LABELS_REPO = ""                          #@param {type:"string"}
HF_LABELS_FILE = "BirdNET_GLOBAL_6K_V2.4_Labels.txt"               #@param {type:"string"}

#@markdown ---
#@markdown ### Labels file settings
#@markdown **Has header row?** — check if the first line of the labels file is a column header (not a label).
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Label column index** — which column contains the label name? (0 = first column, 1 = second, etc.)
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Column delimiter** — how columns are separated in the labels file.
LABELS_DELIMITER = "underscore (_)"  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

#@markdown ---
#@markdown **Activation function** — how the model converts its output to confidence scores.
#@markdown `sigmoid` = each species scored independently (most common). `softmax` = scores sum to 1. `none` = model already outputs probabilities.
ACTIVATION = "sigmoid"  #@param ["sigmoid", "softmax", "none"]

#@markdown ---
#@markdown **Sigmoid sensitivity** — steepness of the sigmoid curve (only used when activation = `sigmoid`). `-1.0` = standard sigmoid; more negative = steeper.
SIGMOID_SENSITIVITY = -1.0  #@param {type:"number"}
#@markdown **Sigmoid bias** — horizontal shift of the sigmoid (only used when activation = `sigmoid`). `1.0` = no shift; above 1.0 = more sensitive (higher scores); below 1.0 = more conservative (lower scores).
SIGMOID_BIAS = 1.0  #@param {type:"slider", min:0.01, max:1.99, step:0.01}

#@markdown **Sample rate (Hz)** — audio sample rate the model expects.
#@markdown BirdNET = 48000 · Google Perch = 32000 · Custom: check your model documentation.
SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Segment duration (seconds)** — length of each audio chunk fed to the model.
#@markdown BirdNET = 3.0 · Google Perch = 5.0 · Custom: check your model documentation.
SEGMENT_DURATION_S = 3.0  #@param {type:"number"}

#@markdown **Segment overlap (0.0–0.9)** — fraction of overlap between consecutive audio chunks.
#@markdown 0.0 = no overlap (default). 0.5 = 50% overlap (twice as many segments, better coverage).
SEGMENT_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

print(f"Model source       : {MODEL_SOURCE}")
print(f"Activation         : {ACTIVATION}")
print(f"Sigmoid      : sensitivity={SIGMOID_SENSITIVITY}  bias={SIGMOID_BIAS}  (used only when activation = sigmoid)")
print(f"Sample rate        : {SAMPLE_RATE} Hz")
print(f"Segment duration   : {SEGMENT_DURATION_S} s")
print(f"Segment overlap    : {SEGMENT_OVERLAP}")
print(f"Labels: column index={LABELS_COLUMN_INDEX}, delimiter='{LABELS_DELIMITER}', header={LABELS_HAS_HEADER}")

In [ ]:
#@title 📍 Recording Site 1 { display-mode: "form" }

#@markdown **Stream ID** — the short code for this recording site (from the optional cell in Step 2).
stream_id_1 = ""  #@param {type:"string"}
#@markdown **Start date and time** (your local time)
min_date_1 = "2025-12-01"  #@param {type:"date"}
min_time_1 = "17:00"  #@param {type:"string"}
#@markdown **End date and time** (your local time)
max_date_1 = "2025-12-01"  #@param {type:"date"}
max_time_1 = "18:00"  #@param {type:"string"}
if stream_id_1.strip():
    print(f"Site 1: '{stream_id_1}' — {min_date_1} {min_time_1} → {max_date_1} {max_time_1}")
else:
    print("Site 1: no stream ID entered.")

In [ ]:
#@title ✅ Confirm recording site list { display-mode: "form" }
#@markdown Run this cell after filling in the site forms above.
#@markdown It will show a summary of all sites that will be analyzed.

import datetime

_empty = ("", "2025-01-01", "00:00", "2025-01-01", "00:00")
try: stream_id_2
except NameError: stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2 = _empty
try: stream_id_3
except NameError: stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3 = _empty
try: stream_id_4
except NameError: stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4 = _empty

def _build_job(stream_id, min_date_str, min_time_str, max_date_str, max_time_str):
    if not stream_id.strip():
        return None
    def _parse(date_str, time_str):
        d = datetime.date.fromisoformat(date_str.strip())
        parts = time_str.strip().split(':')
        h, m = int(parts[0]), int(parts[1])
        s = int(parts[2]) if len(parts) > 2 else 0
        return datetime.datetime(d.year, d.month, d.day, h, m, s)
    return {
        'stream_id':        stream_id.strip(),
        'stream_name':      stream_id.strip(),  # replaced from API below
        'min_date':         _parse(min_date_str, min_time_str),
        'max_date':         _parse(max_date_str, max_time_str),
        'lat':              '',
        'lon':              '',
        'timezone_str':     None,
        'utc_offset_hours': 0,
    }

def _tz_to_utc_offset(timezone_value, reference_dt=None):
    if timezone_value is None or timezone_value == '':
        return None
    try:
        return float(timezone_value)
    except (ValueError, TypeError):
        pass
    try:
        from zoneinfo import ZoneInfo
        import datetime as _dt
        tz = ZoneInfo(str(timezone_value))
        ref = (reference_dt or _dt.datetime.utcnow()).replace(tzinfo=_dt.timezone.utc)
        offset = ref.astimezone(tz).utcoffset()
        return offset.total_seconds() / 3600
    except Exception:
        return None

JOBS = []
for args in [
    (stream_id_1, min_date_1, min_time_1, max_date_1, max_time_1),
    (stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2),
    (stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3),
    (stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4),
]:
    job = _build_job(*args)
    if job:
        JOBS.append(job)

_all_streams = client.streams() or []
_stream_map = {s['id']: s for s in _all_streams if isinstance(s, dict) and 'id' in s}
for job in JOBS:
    s = _stream_map.get(job['stream_id'], {})
    job['stream_name'] = s.get('name', job['stream_id']).replace(' ', '_')
    job['lat'] = s.get('latitude') or s.get('lat', '') or ''
    job['lon'] = s.get('longitude') or s.get('lon', '') or ''
    tz_val = s.get('timezone', None)
    job['timezone_str'] = str(tz_val) if tz_val else None
    mid_dt = job['min_date'] + (job['max_date'] - job['min_date']) / 2
    utc_off = _tz_to_utc_offset(tz_val, reference_dt=mid_dt)
    if utc_off is not None:
        job['utc_offset_hours'] = utc_off
    else:
        print(f"  WARNING: no timezone found for {job['stream_name']} ({job['stream_id']}) — defaulting to UTC+0")
    if not job['lat'] and not job['lon']:
        print(f"  WARNING: no coordinates found for {job['stream_name']} ({job['stream_id']})")

if not JOBS:
    print("WARNING: No recording sites configured. Fill in at least one site form above.")
else:
    print(f"{len(JOBS)} site(s) ready:")
    for j in JOBS:
        utc_off = j['utc_offset_hours']
        print(f"  {j['stream_name']} ({j['stream_id']})")
        print(f"    {j['min_date'].strftime('%Y-%m-%d %H:%M')} → {j['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})  lat={j['lat']} lon={j['lon']}")
    print("\nConfiguration complete. Continue to Step 4.")

### 📍 Additional Recording Sites (optional)
> Need to analyze more than one site? Expand this section to configure up to 3 more recording sites.
> Leave the **Stream ID** empty in any site you don't need. Run gain 'Confirm recording site list' section after adding optional recording sites 

In [ ]:
#@title 📍 Recording Site 2 (optional) { display-mode: "form" }

#@markdown Leave **Stream ID** empty to skip this site.
stream_id_2 = ""  #@param {type:"string"}
min_date_2 = "2025-01-01"  #@param {type:"date"}
min_time_2 = "18:00"  #@param {type:"string"}
max_date_2 = "2025-01-02"  #@param {type:"date"}
max_time_2 = "06:00"  #@param {type:"string"}
if stream_id_2.strip():
    print(f"Site 2: '{stream_id_2}' — {min_date_2} {min_time_2} → {max_date_2} {max_time_2}")
else:
    print("Site 2: empty — will be skipped.")

In [ ]:
#@title 📍 Recording Site 3 (optional) { display-mode: "form" }

#@markdown Leave **Stream ID** empty to skip this site.
stream_id_3 = ""  #@param {type:"string"}
min_date_3 = "2025-01-01"  #@param {type:"date"}
min_time_3 = "18:00"  #@param {type:"string"}
max_date_3 = "2025-01-02"  #@param {type:"date"}
max_time_3 = "06:00"  #@param {type:"string"}
if stream_id_3.strip():
    print(f"Site 3: '{stream_id_3}' — {min_date_3} {min_time_3} → {max_date_3} {max_time_3}")
else:
    print("Site 3: empty — will be skipped.")

In [ ]:
#@title 📍 Recording Site 4 (optional) { display-mode: "form" }

#@markdown Leave **Stream ID** empty to skip this site.
stream_id_4 = ""  #@param {type:"string"}
min_date_4 = "2025-01-01"  #@param {type:"date"}
min_time_4 = "18:00"  #@param {type:"string"}
max_date_4 = "2025-01-02"  #@param {type:"date"}
max_time_4 = "06:00"  #@param {type:"string"}
if stream_id_4.strip():
    print(f"Site 4: '{stream_id_4}' — {min_date_4} {min_time_4} → {max_date_4} {max_time_4}")
else:
    print("Site 4: empty — will be skipped.")

---
## Step 4 — Download audio from Arbimon

This cell connects to Arbimon and downloads all recordings in the time windows you set above.

> Files are downloaded to a **temporary folder inside Colab** (`/content/audio`). They are not saved to your Google Drive. If the session resets, you will need to run this step again.

In [ ]:
#@title 🗑️ Clear downloaded audio (optional) { display-mode: "form" }
#@markdown Run this cell to delete all files in the temporary audio folder and free up Colab disk space.
#@markdown This does **not** affect your Google Drive or any results files.

import shutil, glob as _g

files = _g.glob(f'{AUDIO_DIR}/**/*', recursive=True)
files = [f for f in files if os.path.isfile(f)]

if not files:
    print('Audio folder is already empty.')
else:
    for f in files:
        os.remove(f)
    # Remove empty subdirectories
    for d in sorted(_g.glob(f'{AUDIO_DIR}/*/'), reverse=True):
        try:
            os.rmdir(d)
        except OSError:
            pass
    print(f'Deleted {len(files)} file(s) from {AUDIO_DIR}.')

In [ ]:
#@title ⬇️ Download audio from Arbimon { display-mode: "form" }
#@markdown Downloads all recordings for the sites and time windows configured in Step 3.
#@markdown Files are stored temporarily inside Colab — they are not saved to your Drive.

import os, glob, datetime

os.makedirs(AUDIO_DIR, exist_ok=True)

for job in JOBS:
    utc_off     = job['utc_offset_hours']
    job_utc_min = job['min_date'] - datetime.timedelta(hours=utc_off)
    job_utc_max = job['max_date'] - datetime.timedelta(hours=utc_off)

    print(f"\n{'─' * 65}")
    print(f"Site   : {job['stream_name']}  (id: {job['stream_id']})")
    print(f"Window : {job['min_date'].strftime('%Y-%m-%d %H:%M')} → {job['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})")

    if TIME_FILTER_ENABLED:
        ps = TIME_FILTER_START.split(':')
        pe = TIME_FILTER_END.split(':')
        t_h_s, t_m_s = int(ps[0]), int(ps[1])
        t_h_e, t_m_e = int(pe[0]), int(pe[1])
        crosses_midnight = (t_h_e, t_m_e) < (t_h_s, t_m_s)
        current_day = job['min_date'].date()
        end_day     = job['max_date'].date()
        print(f"Time   : {TIME_FILTER_START}–{TIME_FILTER_END} (local) → downloading per-day windows")
        while current_day <= end_day:
            local_win_s = datetime.datetime(current_day.year, current_day.month, current_day.day, t_h_s, t_m_s)
            if not crosses_midnight:
                local_win_e = datetime.datetime(current_day.year, current_day.month, current_day.day, t_h_e, t_m_e)
            else:
                _nd = current_day + datetime.timedelta(days=1)
                local_win_e = datetime.datetime(_nd.year, _nd.month, _nd.day, t_h_e, t_m_e)
            utc_win_s = max(local_win_s - datetime.timedelta(hours=utc_off), job_utc_min)
            utc_win_e = min(local_win_e - datetime.timedelta(hours=utc_off), job_utc_max)
            if utc_win_s < utc_win_e:
                print(f"  {current_day}  {TIME_FILTER_START}–{TIME_FILTER_END} local → downloading...")
                try:
                    client.download_segments(
                        dest_path=AUDIO_DIR,
                        stream=job['stream_id'],
                        min_date=utc_win_s,
                        max_date=utc_win_e,
                        parallel=True,
                    )
                except Exception as e:
                    print(f"  ERROR: {e}")
            current_day += datetime.timedelta(days=1)
    else:
        print(f"Downloading...")
        try:
            client.download_segments(
                dest_path=AUDIO_DIR,
                stream=job['stream_id'],
                min_date=job_utc_min,
                max_date=job_utc_max,
                parallel=True,
            )
        except Exception as e:
            print(f"  ERROR: {e}")
            print("  Check that the stream_id is correct and you have access to this site.")

all_audio = sorted(
    glob.glob(f'{AUDIO_DIR}/**/*.wav',  recursive=True) +
    glob.glob(f'{AUDIO_DIR}/**/*.flac', recursive=True) +
    glob.glob(f'{AUDIO_DIR}/**/*.mp3',  recursive=True)
)

print(f"\n{'─' * 65}")
print(f"Total audio files downloaded: {len(all_audio)}")
for f in all_audio[:10]:
    print(f"  {f.replace(AUDIO_DIR + '/', '')}")
if len(all_audio) > 10:
    print(f"  ... and {len(all_audio) - 10} more")

---
## Step 5 — Load the model and species labels

This cell loads your model and reads the list of species (or other classes) it can detect.

**Labels file format:** a plain text file with one label per line, for example:
```
Phylodytes luteolus
Dendropsophus minutus
Background noise
```

The number of lines in the labels file must match the number of outputs of your model.

In [ ]:
#@title 🧠 Load model and labels { display-mode: "form" }

import os
import numpy as np

_DELIMITERS = {"tab": "\t", "comma (,)": ",", "semicolon (;)": ";", "underscore (_)": "_"}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Downloading model from HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
    _labels_repo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'Downloading labels from HuggingFace: {_labels_repo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_labels_repo, filename=HF_LABELS_FILE)
elif MODEL_SOURCE == 'google_drive':
    model_path  = DRIVE_MODEL_PATH
    labels_path = DRIVE_LABELS_PATH
else:
    raise ValueError(f"MODEL_SOURCE must be 'huggingface' or 'google_drive', got: {MODEL_SOURCE!r}")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}\nPlease check the path in the Configuration cell (Step 3).")
if not os.path.exists(labels_path):
    raise FileNotFoundError(f"Labels file not found: {labels_path}\nPlease check the path in the Configuration cell (Step 3).")

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'.\nThe model file must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    model = TFLiteInterpreter(model_path=model_path)
    model.allocate_tensors()
    in_shape  = model.get_input_details()[0]['shape']
    out_shape = model.get_output_details()[0]['shape']
    print(f'TFLite model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort
    model = ort.InferenceSession(model_path)
    in_shape  = model.get_inputs()[0].shape
    out_shape = model.get_outputs()[0].shape
    print(f'ONNX model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

if LABELS_HAS_HEADER and lines:
    lines = lines[1:]

for line in lines:
    line = line.strip()
    if not line:
        continue
    if _labels_sep in line:
        parts = line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(parts):
            labels.append(parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f"  WARNING: column index {LABELS_COLUMN_INDEX} not found in line: {line!r} — skipping.")
    else:
        labels.append(line)

print(f'\nLabels loaded: {len(labels)} classes')
print(f'  Delimiter : {LABELS_DELIMITER}  |  Column index : {LABELS_COLUMN_INDEX}  |  Header skipped : {LABELS_HAS_HEADER}')
print(f'  First 5   : {labels[:5]}')
print(f'  Model name: {MODEL_NAME}')

---
## Step 6 — Check already completed analyses

This cell scans your results folder to see which audio files have **already been analyzed**. Those files will be **skipped** in the next step, so you can safely re-run the analysis without repeating work you've already done.

In [ ]:
#@title 🔎 Check { display-mode: "form" }

import os
import re
import glob as _glob
from datetime import datetime, timedelta

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

def parse_file_datetime(filename, utc_offset_hours=0, timezone_str=None):
    name = os.path.basename(filename)
    utc_dt = None
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                          int(m.group(4)), int(m.group(5)), int(m.group(6)))
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                              int(t[:2]), int(t[2:4]), int(t[4:6]))
    if utc_dt is None:
        return None
    if timezone_str:
        try:
            from zoneinfo import ZoneInfo
            local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
            return local_dt.replace(tzinfo=None)
        except Exception:
            pass
    return utc_dt + timedelta(hours=utc_offset_hours)

_job_lookup = {}
for _j in JOBS:
    _job_lookup[_j['stream_id']]   = _j
    _job_lookup[_j['stream_name']] = _j

def _resolve_job(audio_path):
    job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
    if job:
        return job
    for _j in JOBS:
        if _j['stream_id'] and _j['stream_id'] in audio_path:
            return _j
    return JOBS[0] if len(JOBS) == 1 else {}

def get_result_path(audio_path, results_dir, model_name):
    filename  = os.path.basename(audio_path)
    job       = _resolve_job(audio_path)
    site_name = job.get('stream_name', os.path.basename(os.path.dirname(audio_path)))
    utc_off   = job.get('utc_offset_hours', 0)
    tz_str    = job.get('timezone_str', None)
    file_dt   = parse_file_datetime(filename, utc_offset_hours=utc_off, timezone_str=tz_str)
    if file_dt is not None:
        dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
        result_fn = f"{site_name}_{dt_str}.{model_name}.results.txt"
    else:
        file_base = os.path.splitext(filename)[0]
        result_fn = f"{site_name}_{file_base}.{model_name}.results.txt"
    return os.path.join(results_dir, site_name, result_fn)

all_audio = sorted(
    _glob.glob(f'{AUDIO_DIR}/**/*.wav',  recursive=True) +
    _glob.glob(f'{AUDIO_DIR}/**/*.flac', recursive=True) +
    _glob.glob(f'{AUDIO_DIR}/**/*.mp3',  recursive=True)
)

def _time_filter_ok(audio_path):
    if not TIME_FILTER_ENABLED:
        return True
    _job  = _resolve_job(audio_path)
    _fdt  = parse_file_datetime(
        os.path.basename(audio_path),
        utc_offset_hours=_job.get('utc_offset_hours', 0),
        timezone_str=_job.get('timezone_str', None)
    )
    if _fdt is None:
        return True
    from datetime import time as _dtime
    _t  = _fdt.time()
    _ps = TIME_FILTER_START.split(':')
    _pe = TIME_FILTER_END.split(':')
    _ts = _dtime(int(_ps[0]), int(_ps[1]))
    _te = _dtime(int(_pe[0]), int(_pe[1]))
    return (_ts <= _t <= _te) if _ts <= _te else (_t >= _ts or _t <= _te)

to_process   = [f for f in all_audio
                if not os.path.exists(get_result_path(f, DRIVE_RESULTS_DIR, MODEL_NAME))
                and _time_filter_ok(f)]
already_done = [f for f in all_audio if f not in to_process]

print(f'Results folder : {DRIVE_RESULTS_DIR}')
print(f'Model name     : {MODEL_NAME}')
print()
print(f'Total audio files  : {len(all_audio)}')
print(f'Already analyzed   : {len(already_done)}')
print(f'Remaining to run   : {len(to_process)}')

if already_done:
    print()
    print('Already done (will be skipped):')
    for f in already_done:
        rp = get_result_path(f, DRIVE_RESULTS_DIR, MODEL_NAME)
        print(f"  {os.path.basename(f)}  →  {os.path.basename(rp)}")

if not to_process:
    print()
    print('All files have already been analyzed. Nothing to do!')
    print('If you want to re-analyze, delete the corresponding results files from your Drive first.')
else:
    print()
    print(f'Ready to analyze {len(to_process)} file(s). Proceed to Step 7.')

---
## Step 7 — Run the analysis and save results

This is the main analysis step. For each audio file:
1. The recording is split into short segments (e.g. 3 seconds each)
2. Each segment is fed to the model
3. Detections above the confidence threshold are kept
4. Results are saved immediately to your Google Drive

**Result files** are saved as:
```
DRIVE_RESULTS_DIR / SITE_NAME / audio_file.MODEL_NAME.results.txt
```

Each results file has columns: `stream_id`, `site`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `label`, `confidence`

- `start_time` / `end_time` — offset in seconds from the beginning of the audio file for each detected segment.
- `utc_offset` — the UTC offset (in hours) applied to convert recording timestamps to local time.

> Depending on the number of files and the length of recordings, this step can take a while. Progress is shown below.

In [ ]:
#@title 🚀 Run { display-mode: "form" }

import csv
import re
import time
import numpy as np
import librosa
from datetime import datetime, timedelta

def parse_file_datetime(filename, utc_offset_hours=0, timezone_str=None):
    name = os.path.basename(filename)
    utc_dt = None
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                          int(m.group(4)), int(m.group(5)), int(m.group(6)))
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                              int(t[:2]), int(t[2:4]), int(t[4:6]))
    if utc_dt is None:
        return None
    if timezone_str:
        try:
            from zoneinfo import ZoneInfo
            local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
            return local_dt.replace(tzinfo=None)
        except Exception:
            pass
    return utc_dt + timedelta(hours=utc_offset_hours)

def apply_activation(logits, activation, sensitivity=-1.0, bias=1.0):
    arr = np.array(logits, dtype=np.float32).flatten()
    if activation == 'sigmoid':
        transformed_bias = (bias - 1.0) * 10.0
        return 1.0 / (1.0 + np.exp(sensitivity * np.clip(arr + transformed_bias, -20, 20)))
    elif activation == 'softmax':
        arr = arr - arr.max()
        e = np.exp(arr)
        return e / e.sum()
    return arr

def run_model(model, model_type, audio_segment):
    seg = audio_segment.astype(np.float32)
    if model_type == 'tflite':
        in_det  = model.get_input_details()[0]
        out_det = model.get_output_details()[0]
        try:
            model.set_tensor(in_det['index'], seg.reshape(in_det['shape']))
        except ValueError:
            model.set_tensor(in_det['index'], seg.reshape(1, -1))
        model.invoke()
        return model.get_tensor(out_det['index']).flatten()
    else:
        input_name = model.get_inputs()[0].name
        try:
            logits = model.run(None, {input_name: seg.reshape(1, -1)})[0]
        except Exception:
            logits = model.run(None, {input_name: seg[np.newaxis, :]})[0]
        return np.array(logits).flatten()

def preprocess_audio(audio, sr):
    if FILTER_TYPE != 'none':
        from scipy.signal import butter, sosfilt
        nyq = sr / 2.0
        if FILTER_TYPE == 'lowpass':
            sos = butter(5, min(FILTER_HIGH_HZ, nyq - 1) / nyq, btype='low', output='sos')
        elif FILTER_TYPE == 'highpass':
            sos = butter(5, max(FILTER_LOW_HZ, 1) / nyq, btype='high', output='sos')
        elif FILTER_TYPE == 'bandpass':
            lo = max(FILTER_LOW_HZ, 1) / nyq
            hi = min(FILTER_HIGH_HZ, nyq - 1) / nyq
            sos = butter(5, [lo, hi], btype='band', output='sos')
        audio = sosfilt(sos, audio).astype(np.float32)
    if AUDIO_SPEED != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=AUDIO_SPEED)
    return audio

def _in_time_window(file_dt, start_str, end_str):
    from datetime import time as _dtime
    if file_dt is None:
        return True
    t = file_dt.time()
    ps, pe = start_str.split(':'), end_str.split(':')
    t_start = _dtime(int(ps[0]), int(ps[1]))
    t_end   = _dtime(int(pe[0]), int(pe[1]))
    if t_start <= t_end:
        return t_start <= t <= t_end
    return t >= t_start or t <= t_end

_job_lookup = {}
for _j in JOBS:
    _job_lookup[_j['stream_id']]   = _j
    _job_lookup[_j['stream_name']] = _j

def _resolve_job(audio_path):
    job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
    if job:
        return job
    for _j in JOBS:
        if _j['stream_id'] and _j['stream_id'] in audio_path:
            return _j
    return JOBS[0] if len(JOBS) == 1 else {}

segment_samples  = int(SEGMENT_DURATION_S * SAMPLE_RATE)
stride_samples   = max(1, int(segment_samples * (1 - SEGMENT_OVERLAP)))
total_detections = 0
total_start      = time.time()

if not to_process:
    print('No files to process. All analyses are already complete.')
else:
    print(f'Starting analysis of {len(to_process)} file(s) with model "{MODEL_NAME}"')
    print(f'Confidence threshold : {MIN_CONFIDENCE}')
    print(f'Top K                : {TOP_K}')
    print(f'Segment duration     : {SEGMENT_DURATION_S}s  |  Sample rate: {SAMPLE_RATE} Hz')
    if FILTER_TYPE != 'none' or AUDIO_SPEED != 1.0:
        _pp = []
        if FILTER_TYPE != 'none': _pp.append(f'filter={FILTER_TYPE}')
        if AUDIO_SPEED != 1.0:   _pp.append(f'speed={AUDIO_SPEED}x')
        print(f'Preprocessing        : {", ".join(_pp)}')
    print('=' * 65)

    for file_idx, audio_path in enumerate(to_process, 1):
        filename        = os.path.basename(audio_path)
        _job            = _resolve_job(audio_path)
        site_name       = _job.get('stream_name', os.path.basename(os.path.dirname(audio_path)))
        stream_id       = _job.get('stream_id', '')
        site_lat        = _job.get('lat', '')
        site_lon        = _job.get('lon', '')
        site_utc_offset = _job.get('utc_offset_hours', 0)
        tz_str          = _job.get('timezone_str', None)
        utc_offset_str  = f'UTC{site_utc_offset:+.0f}'
        file_dt         = parse_file_datetime(filename, utc_offset_hours=site_utc_offset, timezone_str=tz_str)
        if TIME_FILTER_ENABLED and not _in_time_window(file_dt, TIME_FILTER_START, TIME_FILTER_END):
            print(f'[{file_idx}/{len(to_process)}] {filename}  — skipped (outside {TIME_FILTER_START}–{TIME_FILTER_END})')
            continue

        if file_dt is not None:
            dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
            result_fn = f"{site_name}_{dt_str}.{MODEL_NAME}.results.txt"
        else:
            file_base = os.path.splitext(filename)[0]
            result_fn = f"{site_name}_{file_base}.{MODEL_NAME}.results.txt"
            print(f"  WARNING: could not parse datetime from '{filename}' — using filename as fallback.")

        print(f'[{file_idx}/{len(to_process)}] {filename}  (site: {site_name}, stream: {stream_id})')

        try:
            audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        except Exception as e:
            print(f'  ERROR loading audio: {e} — skipping.')
            continue

        audio = preprocess_audio(audio, SAMPLE_RATE)

        rows        = []
        file_start  = time.time()
        n_segments  = 0

        for start_samp in range(0, len(audio), stride_samples):
            seg = audio[start_samp:start_samp + segment_samples]
            if len(seg) < segment_samples * 0.5:
                continue
            if len(seg) < segment_samples:
                seg = np.pad(seg, (0, segment_samples - len(seg)))

            try:
                logits = run_model(model, MODEL_TYPE, seg)
            except Exception as e:
                print(f'  ERROR during inference at {start_samp/SAMPLE_RATE:.1f}s: {e}')
                continue

            n_segments += 1
            scores     = apply_activation(logits, ACTIVATION, SIGMOID_SENSITIVITY, SIGMOID_BIAS)
            offset_s   = start_samp / SAMPLE_RATE
            end_s      = offset_s + SEGMENT_DURATION_S

            _seg_hits = sorted(
                [(float(scores[i]), i) for i in range(min(len(scores), len(labels)))
                 if float(scores[i]) >= MIN_CONFIDENCE],
                reverse=True
            )[:TOP_K]
            for score, idx in _seg_hits:
                if file_dt is not None:
                    date_str = file_dt.strftime('%Y-%m-%d')
                    time_str = file_dt.strftime('%H:%M:%S')
                else:
                    date_str = ''
                    time_str = ''
                rows.append([stream_id, site_name, site_lat, site_lon, date_str, time_str,
                                 utc_offset_str,
                                 f'{offset_s:.1f}', f'{end_s:.1f}',
                                 labels[idx], f'{float(score):.4f}'])

        elapsed     = time.time() - file_start
        per_seg     = elapsed / n_segments if n_segments else 0
        audio_dur_s = len(audio) / SAMPLE_RATE

        site_dir    = os.path.join(DRIVE_RESULTS_DIR, site_name)
        os.makedirs(site_dir, exist_ok=True)
        result_path = os.path.join(site_dir, result_fn)

        with open(result_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['stream_id', 'site', 'lat', 'lon', 'date', 'time', 'utc_offset', 'start_time', 'end_time', 'label', 'confidence'])
            writer.writerows(rows)

        total_detections += len(rows)
        print(f'  -> {len(rows)} detection(s)  |  {n_segments} segments  |  '
              f'{elapsed:.1f}s ({per_seg*1000:.0f}ms/seg, {audio_dur_s/elapsed:.1f}x realtime)  '
              f'→  {result_fn}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print(f'Analysis complete!')
    print(f'  Files analyzed   : {len(to_process)}')
    print(f'  Total detections : {total_detections}')
    print(f'  Total time       : {total_elapsed:.1f}s')
    print(f'  Results saved to : {DRIVE_RESULTS_DIR}')

---
## Merge Results into a Single CSV

The Run step above saves one result file per recording (so an interrupted run can resume). This step
combines all of those files into a **single CSV** on your Google Drive:
```
RESULTS_FOLDER / MODEL_NAME.merged.results.csv
```
Columns: `stream_id`, `site`, `lat`, `lon`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `label`, `confidence`

Load this file directly in the **Classification Results Analyzer** notebook by pointing its
*Results folder* setting at this `.merged.results.csv` file.

> The individual per-recording files are **not** deleted — you can safely re-run this step at any time.

In [ ]:
#@title 🧩 Merge all results into a single CSV { display-mode: "form" }

#@markdown Combines every per-recording result file in the results folder into one CSV,
#@markdown ready to load in the **Classification Results Analyzer** notebook.
#@markdown The individual per-recording files are kept so a crashed run can still resume.

import os
import csv
import glob

MERGED_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.merged.results.csv')

# Find every per-recording result file written by the Run step (searched recursively, one per site folder).
_result_files = sorted(glob.glob(
    os.path.join(DRIVE_RESULTS_DIR, '**', f'*.{MODEL_NAME}.results.txt'), recursive=True))

if not _result_files:
    print(f'No result files found in: {DRIVE_RESULTS_DIR}')
    print('Run the analysis step first to generate the per-recording results.')
else:
    # Stream file-by-file so we never hold every result in memory at once.
    _header  = None
    _n_files = 0
    _n_rows  = 0
    with open(MERGED_CSV_PATH, 'w', newline='', encoding='utf-8') as _out:
        _writer = csv.writer(_out)
        for _fp in _result_files:
            try:
                with open(_fp, 'r', newline='', encoding='utf-8') as _in:
                    _rows = list(csv.reader(_in))
            except Exception as _e:
                print(f'  WARNING: could not read {os.path.basename(_fp)}: {_e}')
                continue
            if not _rows:
                continue
            _file_header, _data = _rows[0], _rows[1:]
            if _header is None:
                _header = _file_header
                _writer.writerow(_header)
            _writer.writerows(_data)
            _n_files += 1
            _n_rows  += len(_data)

    print('Merge complete!')
    print(f'  Result files merged : {_n_files}')
    print(f'  Total detections    : {_n_rows}')
    print(f'  Merged CSV saved to : {MERGED_CSV_PATH}')